# MACEField: one energy, several responses

[Open this notebook in Colab](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/00_macefield_architecture.ipynb)

This optional introduction explains the field-conditioned energy and checks its polarization derivative using the checkpoint bundled with this repository. It runs on its own; it does not use a checkpoint produced by the training notebook.

For a periodic cell of volume $\Omega$, MACEField defines polarization from one scalar energy:

$$
\mathbf P=-rac{1}{\Omega}rac{\partial E_	heta}{\partialoldsymbol{\mathcal E}}.
$$

The Born effective charge is the position derivative of the cell dipole, and the electronic susceptibility comes from a second field derivative. Forces and response therefore share the same energy surface. The exact implementation is in the [MACEField module on main](https://github.com/mdi-group/mace-field/blob/main/mace/modules/extensions.py).


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def ensure_checkout(path, url):
    path = Path(path)
    if (path / ".git").is_dir():
        branch = subprocess.run(
            ["git", "-C", str(path), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch == "main":
            return
        subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
        subprocess.run(
            ["git", "-C", str(path), "checkout", "-B", "main", "origin/main"],
            check=True,
        )
        return
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", url, str(path)],
        check=True,
    )


if IN_COLAB:
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    PACKAGE_REPO = Path("/content/mace-field")
    ensure_checkout(TUTORIAL_REPO, TUTORIAL_URL)
    ensure_checkout(PACKAGE_REPO, PACKAGE_URL)
else:
    CWD = Path.cwd().resolve()
    TUTORIAL_REPO = next(
        (
            path
            for path in (CWD, *CWD.parents)
            if (path / "tutorials/macefield/data").is_dir()
            and (path / "README.md").is_file()
        ),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run this notebook from a clone of mdi-group/mace-field-tutorial, "
            "or open it in Colab."
        )

    requested_repo = os.environ.get("MACEFIELD_REPO")
    if requested_repo:
        PACKAGE_REPO = Path(requested_repo).expanduser().resolve()
        if not (PACKAGE_REPO / "mace/modules/extensions.py").is_file():
            raise RuntimeError("MACEFIELD_REPO does not point to a MACE-Field checkout.")
        branch = subprocess.run(
            ["git", "-C", str(PACKAGE_REPO), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch != "main":
            raise RuntimeError("Set MACEFIELD_REPO to a checkout on the main branch.")
    else:
        candidates = [
            TUTORIAL_REPO.parent / "mace-field-main",
            TUTORIAL_REPO.parent / "mace-field",
        ]
        PACKAGE_REPO = None
        for candidate in candidates:
            if not (candidate / "mace/modules/extensions.py").is_file():
                continue
            branch = subprocess.run(
                ["git", "-C", str(candidate), "branch", "--show-current"],
                capture_output=True,
                text=True,
                check=False,
            ).stdout.strip()
            if branch == "main":
                PACKAGE_REPO = candidate.resolve()
                break
        if PACKAGE_REPO is None:
            PACKAGE_REPO = Path("/tmp/mace-field-main")
            ensure_checkout(PACKAGE_REPO, PACKAGE_URL)

os.environ["MACEFIELD_REPO"] = str(PACKAGE_REPO)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, str(PACKAGE_REPO))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--editable", str(PACKAGE_REPO)],
    check=True,
)
os.chdir(TUTORIAL_REPO)

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA_DIR = TUTORIAL_DIR / "data"
print("Tutorial repository:", TUTORIAL_REPO)
print("MACE-Field source:", PACKAGE_REPO)


## Inspect the model and check the field derivative

The following finite-difference check verifies the sign and volume normalization of the polarization returned by the calculator. It checks implementation consistency for one teaching-model structure; it is not an accuracy test.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read
from mace.calculators import MACECalculator

MODEL = DATA_DIR / "batio3/MACEField-Landau.model"
SAMPLE = DATA_DIR / "batio3/macefield_batio3_toy_valid.extxyz"

atoms = read(SAMPLE, index=0)
atoms.info.clear()
for key in list(atoms.arrays):
    if key not in {"numbers", "positions"}:
        atoms.arrays.pop(key)

device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
calc = MACECalculator(
    model_paths=str(MODEL),
    model_type="MACEField",
    device=device,
    default_dtype="float32",
    electric_field=[0.0, 0.0, 0.0],
    compute_polarization=True,
    compute_becs=False,
    compute_polarizability=False,
)

field_values = np.linspace(-0.01, 0.01, 9)
energies = []
p_z = []
for field_z in field_values:
    calc.electric_field = [0.0, 0.0, float(field_z)]
    calc.reset()
    atoms.calc = calc
    energies.append(atoms.get_potential_energy())
    p_z.append(float(calc.results["polarization"][2]))

volume = atoms.get_volume()
p_z_fd = -np.gradient(energies, field_values) / volume
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(field_values, p_z, "o-", label="calculator polarization")
ax.plot(field_values, p_z_fd, "s--", label="finite difference of energy")
ax.set(xlabel="electric field Ez (V/Å)", ylabel="Pz (e/Å²)")
ax.grid(alpha=0.25)
ax.legend()
plt.show()
print("maximum absolute difference (e/Å²):", float(np.max(np.abs(np.asarray(p_z) - p_z_fd))))


## Implementation map

The model injects the homogeneous electric field into intermediate equivariant features. The calculator differentiates the scalar output to obtain polarization, Born effective charges, and susceptibility. The checkpoint used in the check above is a five-atom analytic teaching model; its labels are not DFT or experimental data.

Continue with [training BaTiO₃](01_train_batio3.ipynb), or start at any later lesson independently.
